In [55]:
import os
from dotenv import load_dotenv
import requests
from typing import TypedDict
from typing_extensions import Annotated
from langchain.schema import BaseMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq

# ----------------------------
# Load environment variables
# ----------------------------
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")



class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]  # Tracks all messages
    sentiment: str  # Stores sentiment of last user input
    paragraph: str  # Stores generated paragraph


In [56]:
qlm = ChatGroq(model_name="openai/gpt-oss-120b")


In [58]:
def sentiment_node(state: ChatState):

    messages = state["messages"]
    
    # Find last user input
    last_user_msg = None
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            last_user_msg = msg.content
            break
    if last_user_msg is None:
        raise ValueError("No user message found in state")
    
    # Prepare prompt for sentiment detection
    sentiment_prompt = f"""
    Classify the sentiment of the following text as Positive, Negative, or Neutral:
    "{last_user_msg}"
    Output only the sentiment label.
    """
    
    # Use ChatGroq LLaMA 3 model to detect sentiment
    detected_sentiment = qlm.invoke([HumanMessage(content=sentiment_prompt)]).content.strip()
    
    # Update state
    state["sentiment"] = detected_sentiment
    
    # Return in LangGraph node format
    return {"sentiment": detected_sentiment}

In [60]:
gen_llm = ChatGroq(model_name="openai/gpt-oss-120b")

In [61]:
def paragraph_node(state: ChatState):
    messages = state["messages"]
    sentiment = state["sentiment"]  # ✅ use the sentiment from previous node
    
    # Get the last user message (topic)
    last_user_msg = None
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            last_user_msg = msg.content
            break
    if last_user_msg is None:
        raise ValueError("No user message found in state")

    # Prompt for paragraph generation
    generation_prompt = f"""
    Write a detailed and coherent paragraph about the topic:
    "{last_user_msg}"
    The paragraph should reflect a {sentiment.lower()} tone.
    """

    # Use Mixtral (different LLM) to generate
    paragraph = gen_llm.invoke([HumanMessage(content=generation_prompt)]).content.strip()

    # Return updated state
    return {"paragraph": paragraph}

In [62]:
from langgraph.graph import StateGraph, END,START

graph = StateGraph(ChatState)

# 2️⃣ Add nodes (these are your defined functions)
graph.add_node("sentiment_node", sentiment_node)
graph.add_node("paragraph_node", paragraph_node)

In [63]:
graph.add_edge(START, "sentiment_node")
graph.add_edge("sentiment_node", "paragraph_node")
graph.add_edge("paragraph_node", END)

# 4️⃣ Compile graph
app = graph.compile()

In [64]:
from langchain.schema import HumanMessage

user_input = input("Enter your prompt: ")

# Initialize state
chat_state = {
    "messages": [HumanMessage(content=user_input)],
    "sentiment": ""
}

# Invoke workflow and explicitly request paragraph output
result = app.invoke(chat_state, output_keys=["sentiment", "paragraph"])

# Access the outputs safely
print("Sentiment:", result.get("sentiment"))
print("Paragraph:\n", result.get("paragraph", "No paragraph generated"))


Sentiment: Neutral
Paragraph:
 I am a boy, a young person navigating the everyday experiences that shape my identity and perspective. My days are filled with school lessons, friendships, and extracurricular activities that teach me both academic concepts and social skills. I enjoy exploring a variety of interests, from sports and video games to reading and music, each offering a different way to express curiosity and develop competence. At home, I share responsibilities with my family, learning the value of cooperation and routine. While I sometimes encounter challenges—such as balancing homework with leisure or dealing with peer pressure—I also find opportunities to grow through problem‑solving and open communication. Overall, being a boy in my community means participating in a diverse set of routines and relationships that contribute to my personal development and sense of belonging.


In [41]:
print(result)

{'messages': [HumanMessage(content='Talk about hope after loss', additional_kwargs={}, response_metadata={}, id='55909e23-1eab-42a5-8de8-8b5e17f4e743')], 'sentiment': 'Positive'}
